# 05 — Reportes Automáticos
**Proyecto:** Dashboard Predictivo del Mercado Laboral Colombiano
Generación automática de tablas resumen, gráficas y exportación de resultados.


## 1. Imports y configuración

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import os, json
from datetime import datetime

sns.set_theme(style='whitegrid')
os.makedirs('../reports/figures', exist_ok=True)
os.makedirs('../reports', exist_ok=True)

df_serie = pd.read_csv('../data/processed/serie_temporal_td.csv', parse_dates=['fecha'])
df_sena = pd.read_csv('../data/02_intermediate/sena_limpio.csv') if os.path.exists('../data/02_intermediate/sena_limpio.csv') else pd.read_csv('../data/raw/sena/sena_inscritos.csv')
df_pred = pd.read_csv('../data/04_model_output/prediccion_td.csv', parse_dates=['fecha']) if os.path.exists('../data/04_model_output/prediccion_td.csv') else None

print('Datos cargados.' if df_pred is not None else 'Sin predicciones — ejecutar notebook 04 primero.')

## 2. Tabla resumen de indicadores

In [ ]:
resumen = pd.DataFrame({
    'Indicador': ['Tasa Desocupación', 'Tasa Ocupación', 'TGP'],
    'Mínimo (%)': [
        df_serie['tasa_desocupacion'].min(),
        df_serie['tasa_ocupacion'].min(),
        df_serie['tasa_global_participacion'].min()
    ],
    'Máximo (%)': [
        df_serie['tasa_desocupacion'].max(),
        df_serie['tasa_ocupacion'].max(),
        df_serie['tasa_global_participacion'].max()
    ],
    'Promedio (%)': [
        df_serie['tasa_desocupacion'].mean(),
        df_serie['tasa_ocupacion'].mean(),
        df_serie['tasa_global_participacion'].mean()
    ],
    'Último valor (%)': [
        df_serie['tasa_desocupacion'].iloc[-1],
        df_serie['tasa_ocupacion'].iloc[-1],
        df_serie['tasa_global_participacion'].iloc[-1]
    ]
}).round(2)

print('=== TABLA RESUMEN DE INDICADORES LABORALES ===')
resumen

## 3. Dashboard visual completo

In [ ]:
fig = plt.figure(figsize=(16, 12))
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.3)

# Panel 1: Serie temporal TD
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(df_serie['fecha'], df_serie['tasa_desocupacion'], color='#e74c3c', linewidth=2, marker='o', markersize=4, label='TD histórica')
if df_pred is not None:
    ax1.plot(df_pred['fecha'], df_pred['td_predicha'], color='#2980b9', linewidth=2.5, marker='s', markersize=6, label='Predicción IA')
    ax1.fill_between(df_pred['fecha'], df_pred['td_lower'], df_pred['td_upper'], alpha=0.2, color='#2980b9', label='IC 90%')
ax1.set_title('Tasa de Desocupación Colombia + Predicción IA', fontweight='bold')
ax1.set_ylabel('%')
ax1.legend()
ax1.grid(True, alpha=0.4)

# Panel 2: TO y TGP
ax2 = fig.add_subplot(gs[1, 0])
ax2.plot(df_serie['fecha'], df_serie['tasa_ocupacion'], color='#2ecc71', linewidth=2, label='TO')
ax2.plot(df_serie['fecha'], df_serie['tasa_global_participacion'], color='#3498db', linewidth=2, label='TGP')
ax2.set_title('Ocupación y Participación Laboral', fontweight='bold')
ax2.set_ylabel('%')
ax2.legend()
ax2.grid(True, alpha=0.4)

# Panel 3: Top 10 SENA
ax3 = fig.add_subplot(gs[1, 1])
col_ocu = [c for c in df_sena.columns if 'ocupaci' in c.lower()][0]
col_19 = [c for c in df_sena.columns if '2019' in c][0] if any('2019' in c for c in df_sena.columns) else None
col_20 = [c for c in df_sena.columns if '2020' in c][0] if any('2020' in c for c in df_sena.columns) else None
if col_19 and col_20:
    df_sena['_total'] = pd.to_numeric(df_sena[col_19], errors='coerce').fillna(0) + pd.to_numeric(df_sena[col_20], errors='coerce').fillna(0)
    top5 = df_sena.nlargest(5, '_total')
    ax3.barh(range(5), top5['_total'].values, color='#9b59b6', alpha=0.85)
    ax3.set_yticks(range(5))
    ax3.set_yticklabels([str(o)[:30] for o in top5[col_ocu].values], fontsize=8)
    ax3.set_title('Top 5 Ocupaciones SENA', fontweight='bold')
    ax3.set_xlabel('Inscritos')
    ax3.grid(True, alpha=0.4)

fig.suptitle('Dashboard Mercado Laboral Colombia — Datos al Ecosistema 2026',
             fontsize=15, fontweight='bold', y=1.01)
plt.savefig('../reports/figures/dashboard_completo.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dashboard guardado.')

## 4. Exportar reporte JSON de métricas

In [ ]:
reporte = {
    'proyecto': 'Dashboard Predictivo Mercado Laboral Colombia',
    'concurso': 'Datos al Ecosistema 2026',
    'fecha_generado': datetime.now().strftime('%Y-%m-%d %H:%M'),
    'datos': {
        'periodo': f"{df_serie['fecha'].min().strftime('%b %Y')} – {df_serie['fecha'].max().strftime('%b %Y')}",
        'observaciones': len(df_serie),
        'fuentes': 4
    },
    'modelo': {
        'algoritmo': 'Holt-Winters (suavización exponencial triple)',
        'mae_pp': df_pred['mae'].iloc[0] if df_pred is not None else 'N/A',
        'rmse_pp': df_pred['rmse'].iloc[0] if df_pred is not None else 'N/A',
        'horizonte_meses': 6
    },
    'indicadores_actuales': {
        'tasa_desocupacion': float(df_serie['tasa_desocupacion'].iloc[-1]),
        'tasa_ocupacion': float(df_serie['tasa_ocupacion'].iloc[-1]),
        'tgp': float(df_serie['tasa_global_participacion'].iloc[-1])
    },
    'web_app': 'https://economia-empleo.vercel.app',
    'repositorio': 'https://github.com/andresZam12/proyectoMINTIC'
}

with open('../reports/metricas_reporte.json', 'w', encoding='utf-8') as f:
    json.dump(reporte, f, ensure_ascii=False, indent=2)

print('Reporte JSON guardado en reports/metricas_reporte.json')
print(json.dumps(reporte, indent=2, ensure_ascii=False))

## 5. Lista de archivos generados

In [ ]:
print('=== ARCHIVOS GENERADOS ===')
for ruta in ['../reports/figures', '../data/04_model_output', '../models']:
    if os.path.exists(ruta):
        archivos = os.listdir(ruta)
        print(f'\n{ruta}/')
        for f in archivos:
            size = os.path.getsize(os.path.join(ruta, f))
            print(f'  {f} ({size/1024:.1f} KB)')